# Notebook 01 — Softmax and Numerical Stability

**Session:** Construct loss functions with different targets  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement naive Softmax from scratch, observe floating-point overflow on large logits,
and implement max-shifted (`robust_softmax`) normalization so probabilities always sum
to 1 without `OverflowError` or `nan`.

## 1. Naive Softmax and Exponential Overflow

In [ ]:
import math


def naive_softmax(data: list[float]) -> list[float]:
    """Compute Softmax directly; vulnerable to overflow when logits are large."""
    exp_data = [math.exp(x) for x in data]
    sum_exp_data = sum(exp_data)
    return [val / sum_exp_data for val in exp_data]


data1 = [1.0, 2.0, 3.0]
print("naive_softmax([1, 2, 3]):", [round(p, 6) for p in naive_softmax(data1)])

# Large logits trigger math range overflow in IEEE-754 float64
data2 = [1000.0, 1001.0, 1002.0]
try:
    print(naive_softmax(data2))
except OverflowError as exc:
    print(f"naive_softmax({data2}) failed as expected: {exc}")

## 2. Numerically Stable Max-Shifted Softmax

In [ ]:
def robust_softmax(data: list[float]) -> list[float]:
    """Compute shift-invariant Softmax by subtracting max(data) before exp."""
    max_value = max(data)
    shifted = [x - max_value for x in data]
    exp_data = [math.exp(x) for x in shifted]
    sum_exp_data = sum(exp_data)
    return [val / sum_exp_data for val in exp_data]


print("robust_softmax([1, 2, 3]):", [round(p, 7) for p in robust_softmax(data1)])
print("robust_softmax([1000, 1001, 1002]):", [round(p, 7) for p in robust_softmax(data2)])

data3 = [1.0, 1001.0, 1002.0]
probs3 = robust_softmax(data3)
print("robust_softmax([1, 1001, 1002]):", [round(p, 8) for p in probs3])
assert abs(sum(probs3) - 1.0) < 1e-12